Importarea bibliotecilor

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Definirea setului de date

In [2]:
dataset = sns.load_dataset("penguins")
dataset.shape

(344, 7)

In [3]:
dataset.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


# 1. Analiza exploratorie a datelor (EDA)

In [4]:
print("Lista de coloane", dataset.columns.to_list())

Lista de coloane ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']


Verificarea valorilor lipsa

In [5]:
print("Valori lipsa:")
print(dataset.isna().sum())

Valori lipsa:
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64


In [6]:
print("Tipuri de date:")
print(dataset.dtypes)

Tipuri de date:
species               object
island                object
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
dtype: object


In [7]:
dataset.describe(include = "all")

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
count,344,344,342.000000,342.000000,342.000000,342.000000,333
unique,3,3,NaN,NaN,NaN,NaN,2
top,Adelie,Biscoe,NaN,NaN,NaN,NaN,Male
freq,152,168,NaN,NaN,NaN,NaN,168
mean,NaN,NaN,43.921930,17.151170,200.915205,4201.754386,NaN
std,NaN,NaN,5.459584,1.974793,14.061714,801.954536,NaN
min,NaN,NaN,32.100000,13.100000,172.000000,2700.000000,NaN
25%,NaN,NaN,39.225000,15.600000,190.000000,3550.000000,NaN
50%,NaN,NaN,44.450000,17.300000,197.000000,4050.000000,NaN
75%,NaN,NaN,48.500000,18.700000,213.000000,4750.000000,NaN


Afisam distributia variabilei tinta (*target*)

In [8]:
print(dataset['species'].value_counts())

species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64


# 2. Pre-procesarea datelor

In [9]:
numeric_features = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
categorial_features = ['island', 'sex']

Pentru atributele numerice: inlocuim valorile lipsa cu valoarea medie

In [10]:
from sklearn.impute import SimpleImputer
numericalImputer = SimpleImputer(strategy = 'mean')
dataset_num = numericalImputer.fit_transform(dataset[numeric_features])

Pentru atributele numerice: normalizam valorile

In [11]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
dataset_num = scaler.fit_transform(dataset_num)

Aplicam transformarea pe setul initial de date

In [12]:
dataset[numeric_features] = dataset_num
dataset.describe()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
count,3.440000e+02,3.440000e+02,3.440000e+02,3.440000e+02
mean,-1.156697e-15,4.131062e-16,-8.262125e-16,4.131062e-17
std,1.001457e+00,1.001457e+00,1.001457e+00,1.001457e+00
min,-2.174858e+00,-2.060444e+00,-2.065333e+00,-1.880837e+00
25%,-8.548868e-01,-7.889322e-01,-7.796428e-01,-8.162745e-01
50%,6.035444e-02,7.569585e-02,-2.796522e-01,-1.900612e-01
75%,8.422188e-01,7.877425e-01,8.631834e-01,6.866374e-01
max,2.884265e+00,2.211836e+00,2.148873e+00,2.627899e+00


Pentru atributele categoriale: inlocuim valorile lipsa cu Modul (cea frecventa valoare)

In [13]:
categoricalImputer = SimpleImputer(strategy = 'most_frequent')
dataset_cat = categoricalImputer.fit_transform(dataset[categorial_features])
dataset[categorial_features] = dataset_cat
print(dataset.isna().sum())

species              0
island               0
bill_length_mm       0
bill_depth_mm        0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64


Transformarea atributelor categoriale in numerice prin One-Hot Encoding

In [14]:
dataset = pd.get_dummies(dataset, columns = categorial_features)

In [15]:
dataset.head()

,species,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,island_Biscoe,island_Dream,island_Torgersen,sex_Female,sex_Male
0,Adelie,-8.870812e-01,7.877425e-01,-1.422488,-0.565789,False,False,True,False,True
1,Adelie,-8.134940e-01,1.265563e-01,-1.065352,-0.503168,False,False,True,True,False
2,Adelie,-6.663195e-01,4.317192e-01,-0.422507,-1.192003,False,False,True,True,False
3,Adelie,-1.307172e-15,1.806927e-15,0.000000,0.000000,False,False,True,False,True
4,Adelie,-1.328605e+00,1.092905e+00,-0.565361,-0.941517,False,False,True,True,False


# 3. Clasificare binara

In [16]:
dataset_binary = dataset[dataset['species'].isin(['Adelie', 'Gentoo'])].copy()

Definirea vectorul X (de atribute) si y (target)

In [17]:
X = dataset_binary.drop('species', axis = 1)
y = dataset_binary['species'].map({'Adelie': 0, 'Gentoo': 1})

In [18]:
print(X.shape)
print(y.shape)

(276, 9)
(276,)


Impartirea setului de date

In [19]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25)

Definirea si antrenarea modelului

In [20]:
from sklearn.svm import SVC
model = SVC(kernel = 'linear')
model.fit(X_train, y_train)

SVC(kernel='linear')

Determinarea predictilor si evaluarea modelului

In [21]:
y_pred = model.predict(X_test)

In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
accuracy = accuracy_score(y_test, y_pred)
print("Acuratete: ", accuracy)
precision = precision_score(y_test, y_pred)
print("Precizie: ", precision)
recall = recall_score(y_test, y_pred)
print("Recall: ", recall)
f1 = f1_score(y_test, y_pred)
print("Scor F1: ", f1)

Acuratete:  1.0
Precizie:  1.0
Recall:  1.0
Scor F1:  1.0


# 4. Clasificare multiclasa

Definirea vectorilor X si y si utilizare label encoder pentru vectorul tinta

In [23]:
from sklearn.preprocessing import LabelEncoder
X = dataset.drop(columns = 'species', axis = 1)
encoder = LabelEncoder()
y = encoder.fit_transform(dataset['species'])

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25)

In [25]:
model = SVC(kernel = 'poly')
model.fit(X_train, y_train)

SVC(kernel='poly')

In [26]:
y_pred = model.predict(X_test)

In [27]:
accuracy = accuracy_score(y_test, y_pred)
print("Acuratete: ", accuracy)
precision = precision_score(y_test, y_pred, average = 'macro')
print("Precizie: ", precision)
recall = recall_score(y_test, y_pred, average = 'macro')
print("Recall: ", recall)
f1 = f1_score(y_test, y_pred, average = 'macro')
print("Scor F1: ", f1)

Acuratete:  0.9883720930232558
Precizie:  0.990990990990991
Recall:  0.9907407407407408
Scor F1:  0.9907389542735867


# 5. Optimizarea hiperparametrilor folosind `GridSearchCV`

În clasificarea multiclasă, performanța modelului SVM poate depinde de alegerea kernelului și de valoarea parametrului `C`. Pentru a compara sistematic mai multe variante, se utilizează `GridSearchCV` din Scikit-learn.

Sunt evaluate **6 combinații de hiperparametri**. Criteriul de selecție este acuratețea (`scoring='accuracy'`).

In [28]:
from sklearn.model_selection import GridSearchCV

params_svm = [
    {'kernel': ['rbf'], 'C': [0.1, 1, 10]},
    {'kernel': ['poly'], 'C': [0.1, 1, 10]}
]

In [29]:
grid_svm = GridSearchCV(
    SVC(),
    params_svm,
    scoring='accuracy'
)

grid_svm.fit(X_train, y_train)

GridSearchCV(estimator=SVC(),
             param_grid=[{'C': [0.1, 1, 10], 'kernel': ['rbf']},
                         {'C': [0.1, 1, 10], 'kernel': ['poly']}],
             scoring='accuracy')

### Afișarea celor mai buni hiperparametri

Atributul `best_params_` conține combinația de hiperparametri care a obținut cel mai bun scor mediu în validarea încrucișată. Atributul `best_score_` indică valoarea acestui scor.

In [30]:
print("Best SVM parameters:", grid_svm.best_params_)
print("Best cross-validation accuracy:", grid_svm.best_score_)

Best SVM parameters: {'C': 1, 'kernel': 'rbf'}
Best cross-validation accuracy: 0.9921568627450981


### Antrenarea modelului cu cei mai buni hiperparametri

In [31]:
best_svm = SVC(**grid_svm.best_params_)
best_svm.fit(X_train, y_train)

y_pred = best_svm.predict(X_test)

### Evaluarea modelului optimizat

Pentru clasificarea multiclasă, precizia (*precision*), recall și scorul F1 sunt calculate cu `average='macro'`. Această medie calculează mai întâi metrica pentru fiecare clasă și apoi face media aritmetică, acordând aceeași importanță fiecărei clase.

In [32]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy score:", accuracy)

recall = recall_score(y_test, y_pred, average='macro')
print("Recall score:", recall)

precision = precision_score(y_test, y_pred, average='macro')
print("Precision score:", precision)

f1 = f1_score(y_test, y_pred, average='macro')
print("F1 score:", f1)

Accuracy score: 0.9883720930232558
Recall score: 0.9907407407407408
Precision score: 0.990990990990991
F1 score: 0.9907389542735867
